# NLP Lab Assignment 10: Word Embeddings

This notebook follows the assignment PDF: it samples Amazon polarity reviews, preprocesses the review text, compares Word2Vec, GloVe, and FastText sentence vectors, evaluates word analogies, and trains one-hidden-layer sentiment classifiers.

**Run from top to bottom.** The final model cell needs the dataset and all three embedding files. The required files are large and are not included in this assignment folder; set their paths in the configuration cell below. This notebook does not invent scores when those inputs are missing.

## 1. Imports and file paths

Put the downloaded files anywhere convenient and update the paths below. The Amazon CSVs are headerless `label,title,text` files. Google News Word2Vec should be the binary `.bin`, GloVe should be a text vector file, and FastText should be `wiki-news-300d-1M-subword.vec`.

In [1]:
from pathlib import Path
import sys, json, subprocess
import pandas as pd
from IPython.display import display

# Resolve the assignment folder whether Jupyter starts at the repo root or here.
CWD = Path.cwd().resolve()
ASSIGNMENT_DIR = CWD if CWD.name.lower() == "assignment10" else CWD / "Assignment10"
SCRIPT = ASSIGNMENT_DIR / "run_assignment10.py"

# EDIT these paths to match your downloaded files.
TRAIN_CSV = ASSIGNMENT_DIR / "data" / "train.csv"
TEST_CSV = ASSIGNMENT_DIR / "data" / "test.csv"
WORD2VEC_FILE = ASSIGNMENT_DIR / "embeddings" / "GoogleNews-vectors-negative300.bin"
GLOVE_FILE = ASSIGNMENT_DIR / "embeddings" / "glove.6B.300d.txt"
FASTTEXT_FILE = ASSIGNMENT_DIR / "embeddings" / "wiki-news-300d-1M-subword.vec"
ANALOGY_FILE = ASSIGNMENT_DIR / "data" / "questions-words.txt"
OUTPUT_DIR = ASSIGNMENT_DIR / "results"
print("Assignment folder:", ASSIGNMENT_DIR)
print("Implementation script exists:", SCRIPT.exists())

Assignment folder: C:\Users\Lenovo\NLP_Assignments\Assignment10
Implementation script exists: True


## 2. Check required inputs

The assignment uses stratified samples of 10,000 training reviews and 1,000 test reviews. The program uses the first 10,000/1,000 only when the source is smaller; in the normal Kaggle dataset there are enough rows. The analogy questions file downloads automatically if it is not supplied locally.

In [2]:
required_files = {
    "train.csv": TRAIN_CSV,
    "test.csv": TEST_CSV,
    "Google News Word2Vec": WORD2VEC_FILE,
    "GloVe vectors": GLOVE_FILE,
    "FastText vectors": FASTTEXT_FILE,
}
missing = [f"{label}: {path}" for label, path in required_files.items() if not path.is_file()]
if missing:
    print("Inputs still needed:")
    for item in missing:
        print(" -", item)
    print("\nUpdate the paths in Section 1, then rerun this cell and the remaining cells.")
else:
    print("All data and embedding files are present. Ready to run the experiment.")

Inputs still needed:
 - train.csv: C:\Users\Lenovo\NLP_Assignments\Assignment10\data\train.csv
 - test.csv: C:\Users\Lenovo\NLP_Assignments\Assignment10\data\test.csv
 - Google News Word2Vec: C:\Users\Lenovo\NLP_Assignments\Assignment10\embeddings\GoogleNews-vectors-negative300.bin
 - GloVe vectors: C:\Users\Lenovo\NLP_Assignments\Assignment10\embeddings\glove.6B.300d.txt
 - FastText vectors: C:\Users\Lenovo\NLP_Assignments\Assignment10\embeddings\wiki-news-300d-1M-subword.vec

Update the paths in Section 1, then rerun this cell and the remaining cells.


## 3. Run preprocessing, embeddings, analogy evaluation, and classifiers

Preprocessing lowercases text, removes punctuation and digits, tokenizes, and removes English stopwords. TF-IDF is fitted on training reviews only. For each embedding, the script evaluates simple average vectors and sentence-specific TF-IDF weighted averages. Sentiment scores are measured on the held-out 1,000 review sample.

In [3]:
if missing:
    print("Experiment not run: add the required input files first. No performance scores have been generated.")
else:
    command = [
        sys.executable, str(SCRIPT),
        "--train", str(TRAIN_CSV), "--test", str(TEST_CSV),
        "--word2vec", str(WORD2VEC_FILE), "--glove", str(GLOVE_FILE),
        "--fasttext", str(FASTTEXT_FILE), "--analogies", str(ANALOGY_FILE),
    ]
    completed = subprocess.run(command, cwd=ASSIGNMENT_DIR.parent, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"Experiment script exited with status {completed.returncode}")

Experiment not run: add the required input files first. No performance scores have been generated.


## 4. Sentiment results

`macro_f1` is the unweighted mean of the F1 scores for the sentiment classes, so each class contributes equally even if the sampled counts differ. Higher is better. Compare the two representation rows within an embedding to see whether TF-IDF weighting helped; compare embeddings within a representation to see which vector source performed best. Accuracy is the fraction of correctly classified reviews. The six rows are three embeddings ? two sentence representations.

In [4]:
sentiment_path = OUTPUT_DIR / "sentiment_results.csv"
if sentiment_path.exists():
    sentiment_results = pd.read_csv(sentiment_path).sort_values(["representation", "macro_f1"], ascending=[True, False])
    display(sentiment_results)
else:
    print("No sentiment results yet. Run Section 3 after providing the data and embedding files.")

No sentiment results yet. Run Section 3 after providing the data and embedding files.


## 5. Analogy results

The analogy task predicts `b - a + c` and checks whether the nearest vocabulary word matches the expected answer. Overall accuracy is the number correct divided by the number of analogies whose four words are present in that model's vocabulary. Per-category accuracy helps show where a model succeeds or struggles. This vocabulary coverage can differ by embedding, so check the `evaluated` count alongside accuracy. The JSON also includes up to 30 example predictions per embedding.

In [5]:
analogy_path = OUTPUT_DIR / "analogy_results.json"
if analogy_path.exists():
    analogy_results = json.loads(analogy_path.read_text(encoding="utf-8"))
    summary = pd.DataFrame([
        {"embedding": name, "overall_accuracy": value["accuracy"],
         "correct": value["correct"], "evaluated": value["evaluated"]}
        for name, value in analogy_results.items()
    ]).sort_values("overall_accuracy", ascending=False)
    display(summary)
    for name, value in analogy_results.items():
        print(f"\n{name}: category accuracies")
        display(pd.DataFrame(value["categories"])[["category", "accuracy", "evaluated"]])
        print("Example predictions:")
        display(pd.DataFrame(value["examples"]).head(10))
else:
    print("No analogy results yet. Run Section 3 after providing the data and embedding files.")

No analogy results yet. Run Section 3 after providing the data and embedding files.


## 6. What to report

Use the generated tables to report (1) macro F1 for all six sentiment models, (2) whether TF-IDF weighted sentence vectors improved over plain averaging for each embedding, (3) the best embedding/representation combination, and (4) overall plus category-level analogy accuracies with evaluated counts. Describe the observed values from your run; they depend on the downloaded vector versions and should not be filled in before the experiment runs.